# 02 — Baseline ML: Watershed & KMeans Segmentation

This notebook evaluates two classical (non-learning) segmentation baselines on
**100 high-density images** from COCO val2017. We compare each method's predicted
object count against the ground-truth annotation count.

The goal is to establish a **performance floor** that motivates the use of
deep-learning approaches in later phases of this project.

---
## Setup

Import libraries, load COCO annotations, and select 100 dense images for evaluation.

In [1]:
import sys, os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from PIL import Image
from tqdm import tqdm
import random

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.data_loader import get_dense_images, load_image_and_masks
from src.baseline import watershed_segmentation, kmeans_color_segmentation

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Load COCO annotations
ann_file = '../data/annotations/annotations/instances_val2017.json'
coco = COCO(ann_file)

# Get dense images (5-50 objects)
dense_ids = get_dense_images(coco, min_obj=5, max_obj=50)
print(f'Total dense images: {len(dense_ids)}')

# Sample 100 for evaluation
random.seed(42)
eval_ids = random.sample(dense_ids, min(100, len(dense_ids)))
print(f'Selected {len(eval_ids)} images for evaluation.')

loading annotations into memory...


Done (t=0.42s)
creating index...
index created!
Total dense images: 2463
Selected 100 images for evaluation.


---
## Run Both Baselines

For each of the 100 selected images, we run both Watershed and KMeans segmentation.
We record the predicted object count from each method alongside the ground-truth
annotation count.

**Watershed** works by finding intensity valleys between objects — it relies on
clear brightness gradients separating distinct objects.

**KMeans** clusters pixels by colour similarity and then counts connected regions
within each cluster. It assumes that different objects have different colours.

In [3]:
img_dir = '../data/images/val2017'

results = []
for i, img_id in enumerate(eval_ids):
    image, anns = load_image_and_masks(coco, img_id, img_dir)
    gt_count = len(anns)

    # Watershed
    ws_count = watershed_segmentation(image)

    # KMeans
    km_count = kmeans_color_segmentation(image, k=5)

    results.append({
        'image_id': int(img_id),
        'gt_count': gt_count,
        'watershed_count': ws_count,
        'kmeans_count': km_count,
    })

    if (i + 1) % 10 == 0:
        print(f'Processing {i+1}/{len(eval_ids)}...')

print(f'\nDone! Processed {len(results)} images.')

Processing 10/100...


Processing 20/100...


Processing 30/100...


Processing 40/100...


Processing 50/100...


Processing 60/100...


Processing 70/100...


Processing 80/100...


Processing 90/100...


Processing 100/100...

Done! Processed 100 images.


---
## Scatter Plot: Predicted vs Actual Count

A perfect method would have all points on the diagonal (y = x).
Points above the diagonal indicate **over-segmentation** (predicting more objects than exist),
while points below indicate **under-segmentation** (missing objects).

In [4]:
gt = [r['gt_count'] for r in results]
ws = [r['watershed_count'] for r in results]
km = [r['kmeans_count'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Watershed
axes[0].scatter(gt, ws, alpha=0.6, color='#2196F3', edgecolor='white', s=50)
max_val = max(max(gt), max(ws)) + 5
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect (y=x)')
axes[0].set_xlabel('Ground Truth Count', fontsize=12)
axes[0].set_ylabel('Predicted Count', fontsize=12)
axes[0].set_title('Watershed: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# KMeans
axes[1].scatter(gt, km, alpha=0.6, color='#FF9800', edgecolor='white', s=50)
max_val2 = max(max(gt), max(km)) + 5
axes[1].plot([0, max_val2], [0, max_val2], 'r--', linewidth=1.5, label='Perfect (y=x)')
axes[1].set_xlabel('Ground Truth Count', fontsize=12)
axes[1].set_ylabel('Predicted Count', fontsize=12)
axes[1].set_title('KMeans: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('../results/figures/baseline_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/baseline_evaluation.png')

Saved: results/figures/baseline_evaluation.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_60719/546830761.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Accuracy Metrics

We define **count accuracy** as the fraction of images where the predicted count is
within ±3 of the ground truth. We also compute mean absolute error (MAE).

In [5]:
# Compute metrics
gt_arr = np.array(gt)
ws_arr = np.array(ws)
km_arr = np.array(km)

ws_mae = np.mean(np.abs(ws_arr - gt_arr))
km_mae = np.mean(np.abs(km_arr - gt_arr))

ws_acc = np.mean(np.abs(ws_arr - gt_arr) <= 3) * 100
km_acc = np.mean(np.abs(km_arr - gt_arr) <= 3) * 100

print(f'=== Watershed ===')
print(f'  Avg Predicted Count: {ws_arr.mean():.1f}')
print(f'  Avg GT Count:       {gt_arr.mean():.1f}')
print(f'  MAE:                {ws_mae:.2f}')
print(f'  Accuracy (±3):      {ws_acc:.1f}%')
print()
print(f'=== KMeans ===')
print(f'  Avg Predicted Count: {km_arr.mean():.1f}')
print(f'  Avg GT Count:       {gt_arr.mean():.1f}')
print(f'  MAE:                {km_mae:.2f}')
print(f'  Accuracy (±3):      {km_acc:.1f}%')

=== Watershed ===
  Avg Predicted Count: 4.4
  Avg GT Count:       12.0
  MAE:                8.47
  Accuracy (±3):      24.0%

=== KMeans ===
  Avg Predicted Count: 127.5
  Avg GT Count:       12.0
  MAE:                115.51
  Accuracy (±3):      0.0%


---
## Failure Cases & Success Cases

### Failure Cases
We select 3 images where the Watershed prediction is farthest from the ground truth.
These highlight scenarios where classical methods break down.

### Success Cases
We also show 3 images where Watershed was closest to the ground truth, to understand
what properties make an image easier for classical segmentation.

In [6]:
# Sort by Watershed error
errors = [(r, abs(r['watershed_count'] - r['gt_count'])) for r in results]
errors.sort(key=lambda x: x[1], reverse=True)

# Top 3 failures
failures = [e[0] for e in errors[:3]]
# Top 3 successes (closest)
successes = [e[0] for e in errors[-3:]]

print('Failure cases (highest error):')
for f in failures:
    print(f'  Image {f["image_id"]}: GT={f["gt_count"]}, WS={f["watershed_count"]}, KM={f["kmeans_count"]}')
print()
print('Success cases (lowest error):')
for s in successes:
    print(f'  Image {s["image_id"]}: GT={s["gt_count"]}, WS={s["watershed_count"]}, KM={s["kmeans_count"]}')

Failure cases (highest error):
  Image 139099: GT=29, WS=1, KM=266
  Image 570688: GT=35, WS=8, KM=93
  Image 410650: GT=28, WS=2, KM=194

Success cases (lowest error):
  Image 569976: GT=7, WS=7, KM=76
  Image 559547: GT=6, WS=6, KM=194
  Image 234607: GT=5, WS=5, KM=98


In [7]:
# Visualise failure cases
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, r in enumerate(failures):
    image, anns = load_image_and_masks(coco, r['image_id'], img_dir)
    axes[idx].imshow(image)
    axes[idx].set_title(
        f'FAILURE — ID: {r["image_id"]}\n'
        f'GT={r["gt_count"]} | WS={r["watershed_count"]} | KM={r["kmeans_count"]}',
        fontsize=10, fontweight='bold', color='red'
    )
    axes[idx].axis('off')

plt.suptitle('Top 3 Failure Cases (Watershed)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/baseline_failures.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/baseline_failures.png')

Saved: results/figures/baseline_failures.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_60719/975946258.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Visualise success cases
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, r in enumerate(successes):
    image, anns = load_image_and_masks(coco, r['image_id'], img_dir)
    axes[idx].imshow(image)
    axes[idx].set_title(
        f'SUCCESS — ID: {r["image_id"]}\n'
        f'GT={r["gt_count"]} | WS={r["watershed_count"]} | KM={r["kmeans_count"]}',
        fontsize=10, fontweight='bold', color='green'
    )
    axes[idx].axis('off')

plt.suptitle('Top 3 Success Cases (Watershed)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/baseline_successes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/baseline_successes.png')

Saved: results/figures/baseline_successes.png


/var/folders/ym/l1ns8c3x59s96ttvy6dcf_h40000gn/T/ipykernel_60719/3256633780.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Save Results

In [9]:
# Save all results to JSON
baseline_results = {
    'num_images_evaluated': len(results),
    'watershed': {
        'avg_predicted_count': round(float(ws_arr.mean()), 2),
        'avg_gt_count': round(float(gt_arr.mean()), 2),
        'mae': round(float(ws_mae), 2),
        'accuracy_within_3': round(float(ws_acc), 2),
    },
    'kmeans': {
        'avg_predicted_count': round(float(km_arr.mean()), 2),
        'avg_gt_count': round(float(gt_arr.mean()), 2),
        'mae': round(float(km_mae), 2),
        'accuracy_within_3': round(float(km_acc), 2),
    },
    'per_image_results': results,
}

with open('../results/metrics/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print('Saved: results/metrics/baseline_results.json')
print(json.dumps({k: v for k, v in baseline_results.items() if k != 'per_image_results'}, indent=2))

Saved: results/metrics/baseline_results.json
{
  "num_images_evaluated": 100,
  "watershed": {
    "avg_predicted_count": 4.41,
    "avg_gt_count": 12.04,
    "mae": 8.47,
    "accuracy_within_3": 24.0
  },
  "kmeans": {
    "avg_predicted_count": 127.55,
    "avg_gt_count": 12.04,
    "mae": 115.51,
    "accuracy_within_3": 0.0
  }
}


---
## Why Do These Baselines Fail on Dense Images?

### Connecting Back to EDA Findings

Our EDA in Notebook 01 identified four key challenges in the COCO dataset:
heavy occlusion, scale variation, class imbalance, and boundary truncation.
These challenges directly explain the poor performance of both baselines:

**1. Watershed fails on occluded objects** because it relies on finding clear
intensity valleys (dark boundaries) between objects. When objects overlap,
the boundary gradient disappears and Watershed merges multiple objects into
a single watershed basin. This is why it consistently **under-counts** in
crowded scenes — it cannot split touching objects.

**2. KMeans fails when objects share similar colours.** In a scene with ten
brown chairs around a brown table, KMeans clusters all the brown pixels
together into one cluster. Connected components within that cluster then
counts all those objects as one region. KMeans has no concept of spatial
separation between same-colour objects.

**3. Scale variation defeats fixed-parameter methods.** Both Watershed (fixed
blur kernel) and KMeans (fixed k) use global parameters that cannot adapt to
the mixture of tiny and large objects in the same image. A blur that removes
noise on a large object will erase a small object entirely.

**4. High bias, low variance.** Both methods are extremely simple — they have
very few tuneable parameters. This means they have **high bias**: they cannot
capture the complexity of real-world object boundaries, regardless of how
much data we provide. This is a fundamental limitation of classical methods
and the primary motivation for moving to deep learning in Phase 2.

---
**Conclusion:** Classical methods provide a useful baseline but fundamentally
cannot handle the challenges of high-density object segmentation. Phase 2
will explore HOG+SVM and deep-learning approaches (e.g., Mask R-CNN) that
can learn complex, data-driven features for robust instance segmentation.